# Fake News Detection: Training with Tinker

Fine-tunes a supported LLM via Tinker LoRA on the LIAR2 dataset.

**Run order:** 01 → 02 → this notebook

## 1. Setup

In [1]:
import sys, os, json
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.utils import resample
from collections import defaultdict
from dotenv import load_dotenv
load_dotenv('../.env')
import importlib, config
importlib.reload(config)

from config import TRUTHFULNESS_LABELS, RANDOM_SEED, TINKER_CONFIG, PROMPT_TEMPLATE
from models import TinkerClassifier, ClassBalancer
from data_processor import DataProcessor
from training_utils import TinkerTrainer

np.random.seed(RANDOM_SEED)
print('Imports OK')
print(f'Base model : {TINKER_CONFIG["base_model"]}')
print(f'LoRA rank  : {TINKER_CONFIG["lora_rank"]}')
print(f'Epochs     : {TINKER_CONFIG["epochs"]}')
print(f'Batch size : {TINKER_CONFIG["batch_size"]}')

Imports OK
Base model : nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
LoRA rank  : 8
Epochs     : 6
Batch size : 4


In [2]:
# Connect to Tinker and create LoRA training client.
# Reads TINKER_API_KEY from .env automatically.
# Allocates a GPU on Tinker infrastructure — takes a few seconds.
classifier = TinkerClassifier(
    base_model=TINKER_CONFIG['base_model'],
    prompt_template=PROMPT_TEMPLATE,
)
classifier.connect()
classifier.create_training_client(lora_rank=TINKER_CONFIG['lora_rank'])

tokenizer = classifier.get_tokenizer()
print('Tinker training client ready')
print(classifier.get_model_info())

INFO:models:TinkerClassifier initialized with base_model=nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
INFO:models:Connecting to Tinker...
INFO:tinker.lib.public_interfaces.service_client:ServiceClient initialized for session 61eb0051-dbdf-583f-a349-11b8f8ab1a18
INFO:models:Connected to Tinker service
INFO:models:Creating LoRA training client (rank=8, model=nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16)...
INFO:tinker.lib.public_interfaces.service_client:TrainingClient initialized for model 61eb0051-dbdf-583f-a349-11b8f8ab1a18:train:0
INFO:models:Training client ready


Tinker training client ready
{'base_model': 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16', 'type': 'TinkerClassifier (LoRA)', 'labels': ['barely-true', 'false', 'half-true', 'mostly-true', 'pants-fire', 'true'], 'training_client': 'active', 'sampling_client': 'none'}


## 2. Load & Prepare Data

In [3]:
# Tinker needs raw natural-language text, not word-index sequences.
# Load the cleaned dataset from Phase 1.
artifacts_dir = '../artifacts'
df = pd.read_csv(f'{artifacts_dir}/processed_data.csv')
print(f'Loaded {len(df)} rows')

# Use original statement text so the LLM sees natural language.
texts  = df['statement'].astype(str).tolist()
labels = df['label'].astype(str).str.lower().str.strip().tolist()

unknown = set(labels) - set(TinkerClassifier.LABELS)
if unknown:
    print(f'WARNING: unknown labels: {unknown}')
else:
    print(f'All labels valid: {sorted(set(labels))}')

Loaded 10240 rows
All labels valid: ['barely-true', 'false', 'half-true', 'mostly-true', 'pants-fire', 'true']


In [4]:
# Oversample minority classes, then do a 70/15/15 split.
label_indices = defaultdict(list)
for i, lbl in enumerate(labels):
    label_indices[lbl].append(i)

max_count = max(len(v) for v in label_indices.values())
balanced_indices = []
for lbl, idxs in label_indices.items():
    resampled = resample(idxs, n_samples=max_count, replace=True,
                         random_state=RANDOM_SEED)
    balanced_indices.extend(resampled)

np.random.shuffle(balanced_indices)
texts_bal  = [texts[i]  for i in balanced_indices]
labels_bal = [labels[i] for i in balanced_indices]
print(f'Balanced: {len(texts_bal)} samples ({max_count} per class)')

n       = len(texts_bal)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

texts_train  = texts_bal[:n_train]
labels_train = labels_bal[:n_train]
texts_val    = texts_bal[n_train : n_train + n_val]
labels_val   = labels_bal[n_train : n_train + n_val]
texts_test   = texts_bal[n_train + n_val:]
labels_test  = labels_bal[n_train + n_val:]
print(f'Split  -> Train:{len(texts_train)}  Val:{len(texts_val)}  Test:{len(texts_test)}')

Balanced: 12684 samples (2114 per class)
Split  -> Train:8878  Val:1902  Test:1904


In [5]:
# Tokenise each example into a Tinker Datum.
# Prompt tokens get weight 0 (not trained), completion tokens get weight 1.
print('Building training datums...', flush=True)
train_datums = DataProcessor.prepare_tinker_dataset(
    texts_train, labels_train, tokenizer, PROMPT_TEMPLATE)

print('Building validation datums...', flush=True)
val_datums = DataProcessor.prepare_tinker_dataset(
    texts_val, labels_val, tokenizer, PROMPT_TEMPLATE)

print(f'Done. Train={len(train_datums)}  Val={len(val_datums)}', flush=True)

# Quick sanity check
d = train_datums[0]
print(f'Example final weights : {list(d.loss_fn_inputs["weights"])[-6:]}')

Building training datums...


INFO:data_processor:Prepared 8878 Tinker datums


Building validation datums...


INFO:data_processor:Prepared 1902 Tinker datums


Done. Train=8878  Val=1902
Example final weights : [('data', [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1]), ('dtype', 'float32'), ('shape', None)]


## 3. Tinker LoRA Training

In [6]:
import importlib
import sys
sys.path.insert(0, '../src')
import models
importlib.reload(models)

test_uri = classifier.save_for_inference(name='fake-news-liar2')
print(test_uri)


INFO:models:Saving weights as 'fake-news-liar2'...
INFO:models:Weights saved. URI: 61eb0051-dbdf-583f-a349-11b8f8ab1a18:train:0


61eb0051-dbdf-583f-a349-11b8f8ab1a18:train:0


In [7]:
tc = classifier.training_client
print('model_id:', tc.model_id)
print('_guaranteed_model_id:', tc._guaranteed_model_id)
print('holder session_id:', tc.holder.get_session_id())


model_id: 61eb0051-dbdf-583f-a349-11b8f8ab1a18:train:0
_guaranteed_model_id: <bound method TrainingClient._guaranteed_model_id of <tinker.TrainingClient object at 0x0000021A86D41670>>
holder session_id: 61eb0051-dbdf-583f-a349-11b8f8ab1a18


In [8]:
import importlib, sys
sys.path.insert(0, '../src')
import models
importlib.reload(models)

uri = classifier.save_for_inference(name='fake-news-liar2')
print('URI:', uri)


INFO:models:Saving weights as 'fake-news-liar2'...
INFO:models:Weights saved. URI: 61eb0051-dbdf-583f-a349-11b8f8ab1a18:train:0


URI: 61eb0051-dbdf-583f-a349-11b8f8ab1a18:train:0


In [11]:
trainer = TinkerTrainer(
    classifier=classifier,
    learning_rate=TINKER_CONFIG['learning_rate'],
    batch_size=TINKER_CONFIG['batch_size'],
)

history = trainer.train(
    train_datums=train_datums,
    val_datums=val_datums,
    epochs=TINKER_CONFIG['epochs'],
)

print('Training history:')
for ep, (tl, vl) in enumerate(
        zip(history['train_loss'], history['val_loss']), 1):
    print(f'  Epoch {ep}: train={tl:.4f}  val={vl:.4f}')

INFO:training_utils:Starting Tinker training: 6 epochs, 8878 train / 1902 val datums, batch_size=4, lr=2e-05
INFO:training_utils:
Epoch 1/6
INFO:training_utils:  Batch 10/2220  loss=1.2151
INFO:training_utils:  Batch 20/2220  loss=1.2527
INFO:training_utils:  Batch 30/2220  loss=0.6622
INFO:training_utils:  Batch 40/2220  loss=0.7609
INFO:training_utils:  Batch 50/2220  loss=0.9028
INFO:training_utils:  Batch 60/2220  loss=0.7282


KeyboardInterrupt: 

In [ ]:
# Save weights and get sampling client for inference.
classifier.sampling_client = classifier.training_client.save_weights_and_get_sampling_client(
    name="fake-news-liar2"
)
tinker_uri = classifier.training_client.holder.get_session_id() + ":train:0"
print(f"Tinker URI: {tinker_uri}")

uri_path = f"{artifacts_dir}/tinker_weights_uri.txt"
with open(uri_path, "w") as f:
    f.write(tinker_uri)
print(f"URI saved to {uri_path}")


In [ ]:
plt.figure(figsize=(8, 4))
ep = range(1, len(history['train_loss']) + 1)
plt.plot(ep, history['train_loss'], marker='o', label='Train loss')
plt.plot(ep, history['val_loss'],   marker='s', label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss per token')
plt.title('Tinker LoRA Training Curve')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Save Weights

In [ ]:
# Save LoRA weights to Tinker and get a sampling client.
# The URI is written to disk so predict.py can reload the model later.
tinker_uri = classifier.save_for_inference(name='fake-news-liar2')
print(f'Tinker URI: {tinker_uri}')

uri_path = f'{artifacts_dir}/tinker_weights_uri.txt'
with open(uri_path, 'w') as f:
    f.write(tinker_uri)
print(f'URI saved to {uri_path}')

## 5. Test Set Evaluation

In [ ]:
print(f'Predicting {len(texts_test)} test examples...')
test_preds = []
for i, text in enumerate(texts_test):
    pred = classifier.predict(text)
    test_preds.append(pred)
    if (i + 1) % 100 == 0:
        print(f'  {i + 1}/{len(texts_test)}')

label_order = TinkerClassifier.LABELS  # alphabetical
label2idx   = {l: i for i, l in enumerate(label_order)}
y_true = [label2idx.get(l, -1) for l in labels_test]
y_pred = [label2idx.get(l, -1) for l in test_preds]

test_acc = accuracy_score(y_true, y_pred)
test_f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test Macro F1 : {test_f1:.4f}')

## 6. Detailed Evaluation

In [ ]:
print(classification_report(y_true, y_pred,
                            target_names=label_order, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_order, yticklabels=label_order)
plt.title('Confusion Matrix — Tinker LLM')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Save Results

In [ ]:
results = {
    'tinker_llm': {
        'base_model':        TINKER_CONFIG['base_model'],
        'lora_rank':         TINKER_CONFIG['lora_rank'],
        'tinker_uri':        tinker_uri,
        'test_accuracy':     float(test_acc),
        'test_macro_f1':     float(test_f1),
        'train_loss_history': history['train_loss'],
        'val_loss_history':   history['val_loss'],
    }
}
with open(f'{artifacts_dir}/training_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved training_results.json')

## 8. Final Summary

In [ ]:
print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
print(f'  Model      : {TINKER_CONFIG["base_model"]}')
print(f'  LoRA rank  : {TINKER_CONFIG["lora_rank"]}')
print(f'  Test acc   : {test_acc:.4f}')
print(f'  Test F1    : {test_f1:.4f}')
print(f'  URI        : {tinker_uri}')
print('=' * 60)
print()
print('To run inference:')
print('  python src/predict.py --text "Your statement here" --proba')

In [ ]:
import json, os

PREFIX = 'v3'
save_dir = f'{artifacts_dir}/{PREFIX}'
os.makedirs(save_dir, exist_ok=True)

# Save metrics
results = {
    'base_model':         TINKER_CONFIG['base_model'],
    'lora_rank':          TINKER_CONFIG['lora_rank'],
    'learning_rate':      TINKER_CONFIG['learning_rate'],
    'batch_size':         TINKER_CONFIG['batch_size'],
    'epochs_trained':     len(history['train_loss']),
    'tinker_uri':         tinker_uri,
    'test_accuracy':      float(test_acc),
    'test_macro_f1':      float(test_f1),
    'train_loss_history': history['train_loss'],
    'val_loss_history':   history['val_loss'],
}
with open(f'{save_dir}/training_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved {save_dir}/training_results.json')

# Save loss curve
fig1, ax1 = plt.subplots(figsize=(8, 4))
ep = range(1, len(history['train_loss']) + 1)
ax1.plot(ep, history['train_loss'], marker='o', label='Train loss')
ax1.plot(ep, history['val_loss'],   marker='s', label='Val loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title(f'[{PREFIX}] Tinker LoRA Training Curve')
ax1.legend()
fig1.tight_layout()
fig1.savefig(f'{save_dir}/loss_curve.png', dpi=150)
plt.show()
print(f'Saved {save_dir}/loss_curve.png')

# Save confusion matrix
fig2, ax2 = plt.subplots(figsize=(9, 7))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=TinkerClassifier.LABELS,
            yticklabels=TinkerClassifier.LABELS, ax=ax2)
ax2.set_title(f'[{PREFIX}] Confusion Matrix — Tinker LLM')
ax2.set_ylabel('True label')
ax2.set_xlabel('Predicted label')
plt.xticks(rotation=45, ha='right')
fig2.tight_layout()
fig2.savefig(f'{save_dir}/confusion_matrix.png', dpi=150)
plt.show()
print(f'Saved {save_dir}/confusion_matrix.png')
